# Automatic Alternate Abbreviation Annotation

I am using Claude to try to identify alias symbols of genes that are just variations of abbreviations of the official(primary) gene symbols and the official gene name. The goal is to be able to do this automatically instead of manually. 
These symbols represent the gene with a different symbol than the primary gene symbol. For example, the official long gene name of HGNC:5 is “alpha-1-B glycoprotein” with a primary gene symbol of A1BG. One of the aliases is A1B which is found in PMID: 2447112 to represent “Alpha 1-beta glycoprotein”. This is the same official name of the gene but the abbreviation is different, A1B vs A1BG. 


In [ ]:
import re
import requests
import math
from collections.abc import Mapping
from enum import StrEnum
from pathlib import Path
from typing import Any
import time
import ast
import pickle

import polars as pl
from pydantic import BaseModel, ConfigDict
from tqdm.notebook import tqdm
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import BasePromptTemplate, build_empty_registry
from wags_llm.services import StructuredTaskRunner
from rapidfuzz.distance import LCSseq

## Classes (some are in the functions section)

In [2]:
class SkipReason(StrEnum):
    """Reason for why LLM invocation was skipped"""

    HSA_PREFIX = "hsa_prefix"
    EXTRA_CHARACTERS = "extra_characters"
    LOW_LCS_SIMILARITY = "low_lcs_similarity"


class AlternateAbbreviationPredictionResult(BaseModel):
    """Model for LLM and human curator result for determining whether an
    alias symbol corresponds to the primary gene symbol.
    """

    model_config = ConfigDict(extra="forbid", use_enum_values=True)

    llm_annotation: bool | None = None
    llm_skip_reason: SkipReason | None = None
    error_message: str | None = None
    lcs_similarity_score: float | None = None

## Helper Functions

### Pre-Processing

In [3]:
def get_gene_name(hgnc_id):
    """ Retreive official gene name from HGNC for each sample in the set.

    :param hgnc_id: The HGNC ID associated with the primary gene symbol of each sample
    :return: The official gene name from HGNC for the passed HGNC ID
    """
    url = f"https://rest.genenames.org/fetch/hgnc_id/{hgnc_id}"
    headers = {"Accept": "application/json"}  # Request JSON format
    time.sleep(0.5)
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        try:
            gene_name = data['response']['docs'][0]['name']
            return gene_name
        except IndexError:
            return f"No gene found for HGNC ID {hgnc_id}"
    else:
        return f"Error: {response.status_code}"

In [4]:
def lcs_similarity(s1, s2):
    return LCSseq.normalized_similarity(s1, s2)

def has_extra_characters(alias, gene_name):
    norm_alias = re.sub(r'[^A-Z0-9]', '', alias.upper())

    alias_set = set(norm_alias)
    name_set = set(gene_name)

    extra_chars = alias_set - name_set

    return len(extra_chars)

def should_call_llm(alias_symbol, primary_gene_symbol, gene_name, threshold=0.20):
    """Determine whether an alias symbol should be sent to the LLM for further evaluation.

    Applies a series of heuristic filters to the alias symbol
    - checks for excluded prefixes ("HSA-", this is not an alternate abbreviation alias 
    because it is an identifier being used as an alias)
    - extra characters in the alias, not in the gene name
    - longest common subsequence (LCS) similarity to the primary gene symbol to be at least 20%

    :param alias_symbol: The alias gene symbol being evaluated
    :param primary_gene_symbol: The official primary gene symbol
    :param gene_name: The full gene name associated with the gene
    :param threshold: Minimum LCS similarity score required to pass filtering
    :return: A tuple containing:
        - bool: Whether the alias should be sent to the LLM
        - float: The computed LCS similarity score
        - str: The reason code for the decision
    """

    alias = alias_symbol.upper()
    primary = primary_gene_symbol.upper()
    name = gene_name.upper()

    if alias.startswith("HSA-"):
        return False, 0, SkipReason.HSA_PREFIX

    extra_count = has_extra_characters(alias_symbol, name)

    # block only if ≥2 extra characters
    if extra_count >= 2:
        return False, 0, SkipReason.EXTRA_CHARACTERS

    score = lcs_similarity(alias, primary)

    if score < threshold:
        return False, score, SkipReason.LOW_LCS_SIMILARITY

    return True, score, None

### LLM

In [5]:
PROMPT_NAME = "alias_symbol_annotation:alternate_abbreviation"
PROMPT_VERSION = "v1"


class AlternateAbbreviationPromptV1(BasePromptTemplate):
    """Version 1 prompt for predicting alternate abbreviation relationships."""

    version = PROMPT_VERSION
    name = PROMPT_NAME

    def build_system_prompt(self) -> str:
        """Build the system prompt for predicting whether an alias is an alternate abbreviation of the primary gene symbol.

        :returns: System prompt text.
        """
        return (
            "Role: You are a biomedical gene nomenclature curator trained in HGNC gene identity and alias symbol resolution.\n"
            "You will get fired if your accuracy is less than 95%.\n"
            "Background:\n"
            "An alternate abbreviation is when an alias symbol represents the same official gene name or the\n"
            "primary gene symbol but with different letters. If the alias symbol is representing a different description or a previous name of\n"
            "the gene then it is not an alternate abbreviation. Be careful with alias symbols that seem to be shortened versions\n"
            "of the primary gene symbol, they may be a family name and therefore not an alternate abbreviation. Alias symbols\n"
            "have extra characters may be alternate abbreviations unless they have characters that are not present in the official\n"
            "gene name provided in the prompt. Keep in mind, a gene name with a number after the name is not the same gene as a gene name\n"
            "with no numbers or different numbers after the name.\n"
            "Task:\n"
            "Determine whether the alias symbol is an abbreviation variant of the official HGNC gene name.\n"
        )

    def build_user_prompt(
        self,
        payload: Mapping[str, Any],
    ) -> str:
        """Build the user prompt for a single alias symbol.

        :param payload: The alias symbol, HGNC ID, primary gene symbol, official gene name to be evaluated, 
        :returns: User prompt text
        """
        return (
            f"Alias Symbol: {payload['gene_symbol']}\n"
            f"Primary Gene Symbol: {payload['primary_gene_symbol']}\n"
            f"Official Gene Name: {payload['gene_name']}\n"
            f"HGNC ID: {payload['hgnc_id']}\n"
        )

In [6]:
def get_alt_abbreviation_annotation(
    task_runner: StructuredTaskRunner,
    gene_symbol: str,
    primary_gene_symbol: str,
    gene_name: str,
    hgnc_id: str,
) -> AlternateAbbreviationPredictionResult:
    """Annotate a single alias relationship."""

    should_call, score, skip_reason = should_call_llm(
        gene_symbol,
        primary_gene_symbol,
        gene_name,
    )

    if not should_call:
        return AlternateAbbreviationPredictionResult(
            llm_skip_reason=skip_reason,
            lcs_similarity_score=score,
        )

    try:
        task_result = task_runner.execute(
            prompt_name=PROMPT_NAME,
            prompt_version=PROMPT_VERSION,
            payload={
                "gene_symbol": gene_symbol,
                "primary_gene_symbol": primary_gene_symbol,
                "gene_name": gene_name,
                "hgnc_id": hgnc_id,
            },
            response_model=AlternateAbbreviationPredictionResult,
        )

    except Exception as e:
        return AlternateAbbreviationPredictionResult(
            error_message=str(e),
            lcs_similarity_score=score,
        )

    annotation = AlternateAbbreviationPredictionResult.model_validate(
        task_result
    )

    return AlternateAbbreviationPredictionResult(
        llm_annotation=annotation.llm_annotation,
        lcs_similarity_score=score,
    )

In [7]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 150
TEMPERATURE = 0.0


def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM alternate abbreviation alias annotator

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(AlternateAbbreviationPromptV1())
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )


task_runner = build_llm_task_runner(
    MODEL_ID, REGION_NAME, PROFILE_NAME, MAX_TOKENS, TEMPERATURE
)

In [8]:
def run_single_pass(df, task_runner):
    results = []

    for row in tqdm(df.iter_rows(named=True), total=df.height):
        result = get_alt_abbreviation_annotation(
            task_runner=task_runner,
            gene_symbol=row["gene_symbol"],
            primary_gene_symbol=row["primary_gene_symbol"],
            gene_name=row["gene_name"],
            hgnc_id=row["HGNC_ID"],
        )
        results.append(result)

    return results

In [9]:
def run_experiments(df, temperatures, num_runs):
    stored_runs = []

    for temp in temperatures:

        task_runner = build_llm_task_runner(
            MODEL_ID,
            REGION_NAME,
            PROFILE_NAME,
            MAX_TOKENS,
            temp,
        )

        for run_idx in range(num_runs):

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for row in tqdm(
                df.iter_rows(named=True),
                total=df.height,
                desc=f"T={temp}, run={run_idx+1}",
                leave=False,
            ):

                result = get_alt_abbreviation_annotation(
                    task_runner=task_runner,
                    gene_symbol=row["gene_symbol"],
                    primary_gene_symbol=row["primary_gene_symbol"],
                    gene_name=row["gene_name"],
                    hgnc_id=row["HGNC_ID"],
                )

                results.append(result)

            stored_runs.append({
                "run_idx": run_idx,
                "temperature": temp,
                "results": results,
            })

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs

### Result Analysis

In [10]:
def create_analysis_summary(cm: pl.DataFrame) -> pl.DataFrame:
    """Compute per-class recall-style summary from a confusion matrix.

    :param cm: Confusion matrix (square DataFrame)
    :return: Summary DataFrame per class
    """
    rows = []

    classes = cm.columns[1:]

    for cls in classes:
        row = cm.filter(pl.col("gt") == cls)

        if row.height == 0:
            numerator = 0
            denominator = 0
        else:
            numerator = row.select(pl.col(cls)).item()

            denominator = row.select(pl.sum_horizontal(classes)).item()

        rows.append(
            {
                "consensus_w_curator": cls,
                "numerator": numerator,
                "denominator": denominator,
                "percentage": (
                    (numerator or 0) / denominator * 100
                    if denominator is not None and denominator > 0
                    else 0.0
                ),
            }
        )

    return pl.DataFrame(rows)

In [11]:
def compute_overall_accuracy(cm: pl.DataFrame) -> float:
    """Compute overall accuracy from a confusion matrix.

    :param cm: Confusion matrix DataFrame (square matrix)
    :return: Accuracy as a float between 0 and 1
    """
    classes = cm.columns[1:]

    total = cm.select(pl.sum_horizontal(classes)).to_series().sum()

    correct = 0

    for cls in classes:
        row = cm.filter(pl.col("gt") == cls)

        if row.height > 0:
            correct += row.select(pl.col(cls)).item() or 0

    return correct / total if total > 0 else math.nan

In [12]:
def boolean_confusion_matrix(df: pl.DataFrame, gt_col: str, pred_col: str) -> pl.DataFrame:
    """
    Compute a boolean confusion matrix in the traditional matrix style.

    Parameters:
    - df: pandas DataFrame
    - gt_col: str, name of the ground truth column
    - pred_col: str, name of the predicted column

    Returns:
    - confusion_matrix: pandas DataFrame with row/column labels
    """

    df_clean = (
        df.filter(
            pl.col(gt_col).is_not_null() &
            pl.col(pred_col).is_not_null()
        )
        .with_columns([
            pl.col(gt_col).cast(pl.Boolean).alias("gt"),
            pl.col(pred_col).cast(pl.Boolean).alias("pred"),
        ])
    )

    return (
        df_clean
        .group_by(["gt", "pred"])
        .len()
        .pivot(
            values="len",
            index="gt",
            on="pred",
            aggregate_function="sum",
        )
        .fill_null(0)
    )

In [13]:
def analyze_results(
    df: pl.DataFrame,
) -> tuple[pl.DataFrame, float, pl.DataFrame]:
    """Evaluate LLM predictions against ground truth using a confusion matrix."""

    analysis_df = df.clone()

    cm = boolean_confusion_matrix(
        analysis_df,
        "alternate_abbreviation_status_w_no_context",
        "llm",
    )

    # ---- extract confusion matrix values safely ----
    TN = cm.filter(pl.col("gt") == False)["false"].item()
    FP = cm.filter(pl.col("gt") == False)["true"].item()
    FN = cm.filter(pl.col("gt") == True)["false"].item()
    TP = cm.filter(pl.col("gt") == True)["true"].item()

    # ---- metrics ----
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    metrics_df = pl.DataFrame({
        "precision": [precision],
        "recall": [recall],
        "f1": [f1],
        "TP": [TP],
        "FP": [FP],
        "TN": [TN],
        "FN": [FN],
    })

    match_analysis_summary = create_analysis_summary(cm)
    accuracy = compute_overall_accuracy(cm)

    return match_analysis_summary, accuracy, cm, metrics_df

In [14]:
def process_results(
    df: pl.DataFrame,
) -> tuple[dict, dict, list]:
    """Run analysis on a provided model output DataFrame.

    :param df: Input DataFrame containing model results
    :param collapse: Whether to collapse categories during analysis
    :return: Tuple of all results (summaries, cm, accuracies)
    """
    summary, accuracy, cm, metrics_df = analyze_results(
        df,
    )

    return (
        summary,
        cm,
        accuracy,
        metrics_df
    )

## Add more alias symbols to annotate

Only run this section if you want to add more samples to manually curate

In [15]:
SKIP_ADD_MORE_SAMPLES = True
if not SKIP_ADD_MORE_SAMPLES:
    NUMBER_OF_NEW_SAMPLES_TO_ADD = 0 # Add number of new samples you want to add

In [16]:
ID_COLS = ["HGNC_ID", "ENSG_ID", "NCBI_ID"]

# Load and clean capture df
## This is the file with alias and primary gene symbol pairs and what categories the aliases were captured as
## Generated in the 5_symbol_capture_analysis.ipynb
## Clean it up with converting to booleans and renaming columns for clarity
if not SKIP_ADD_MORE_SAMPLES:
    capture_df = (
        pl.read_csv("../output/summary_df.csv")
        .rename({
            "captured": "captured_status",
            "captured as:": "captured_category_list",
        })
        .drop("")
        .with_columns(
            pl.when(pl.col("captured_status") == "T")
            .then(True)
            .when(pl.col("captured_status") == "F")
            .then(False)
            .otherwise(None)
            .alias("captured_status"),

            pl.col(ID_COLS).map_elements(
                lambda x: (
                    ", ".join(sorted(ast.literal_eval(x)))
                    if x is not None
                    else None
                ),
                return_dtype=pl.String,
            ),
        )
    )

    # Sample new aliases
    ## To add to the sample set for manual annotation (68- wanted to get to a total of 500)

    new_samples_df = (
        capture_df
        .filter(
            (pl.col("captured_category_list") != "Primary Gene Symbol")
            | pl.col("captured_category_list").is_null()
        )
        .group_by("captured_status")
        .map_groups(
            lambda g: g.sample(
                n=min(len(g), (NUMBER_OF_NEW_SAMPLES_TO_ADD/2)),
                seed=41,
            )
        )
    )

    # Load curated annotations

    curated_df = pl.read_excel(
        "/Users/rsaxs014/Desktop/gene-harmony-analysis/output/alt_abbrev_annotation_manually_annotated_df.xlsx"
    )

    # Add gene_name column safely to new samples df

    if "gene_name" not in new_samples_df.columns:
        new_samples_df = new_samples_df.with_columns(pl.lit(None).alias("gene_name"))

    # Make sure the newly added samples don't already exist in the manually curated set

    new_samples_df = (
        new_samples_df
        .join(
            curated_df,
            on=["HGNC_ID", "captured_status"],
            how="anti",
        )
        .with_columns(
            pl.lit(None).alias(
                "alternate_abbreviation_status_w_no_context"
            )
        )
        .select(curated_df.columns)
    )

    # Combine dfs

    df = pl.concat(
        [curated_df, new_samples_df],
        how="vertical",
    )

    # Add gene names, by HGNC ID, to the annotation set for easier manual review

    missing_ids = (
        df
        .filter(pl.col("gene_name").is_null())
        .select("HGNC_ID")
        .unique()
        .to_series()
        .to_list()
    )

    gene_map = {
        hgnc_id: get_gene_name(hgnc_id)
        for hgnc_id in tqdm(missing_ids)
    }

    df = df.with_columns(
        pl.col("gene_name")
        .fill_null(pl.col("HGNC_ID").replace(gene_map))
    )

    # Export

    df.write_excel(
        "../output/alt_abbrev_annotation_to_annotate_df.xlsx"
    )
    f"The original set had {curated_df.height} samples, the new set has {df.height} samples after adding new ones and removing any that were already in the curated set({new_samples_df.height} new samples)"

**After manually annotating the new rows in alt_abbrev_annotation_to_annotate_df.xlsx, it is renamed to alt_abbrev_annotation_manually_annotated_df.xlsx**

## Run LLM with gene symbols, name, and prompt

In [17]:
# Using the manually curated set of samples
SAMPLE_PATH = Path(
    "/Users/rsaxs014/Desktop/gene-harmony-analysis/output/alt_abbrev_annotation_manually_annotated_df.xlsx"
)


In [18]:
df = pl.read_excel(SAMPLE_PATH)

In [19]:
LLM_RUN_RESULTS_DIR = Path("llm_run_results")
LLM_RUN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [20]:
TEMPERATURES = [0.0]
NUM_RUNS = 1

stored_runs_path = Path("stored_runs.pkl")

if stored_runs_path.exists():
    print("Loading existing stored runs...")
    
    with open(stored_runs_path, "rb") as f:
        stored_runs = pickle.load(f)

else:
    print("Running experiments...")
    
    stored_runs = run_experiments(
        df,
        TEMPERATURES,
        NUM_RUNS,
    )

    with open(stored_runs_path, "wb") as f:
        pickle.dump(stored_runs, f)

Running temp=0.0, run=1


T=0.0, run=1:   0%|          | 0/504 [00:00<?, ?it/s]

Done temp=0.0, run=1


In [21]:
rows = []

for run in stored_runs:
    for i, r in enumerate(run["results"]):
        rows.append({
            "row": i,
            "run": run["run_idx"],
            "temp": run["temperature"],
            "llm": r.llm_annotation,
            "alternate_abbreviation_status_w_no_context": df[i, "alternate_abbreviation_status_w_no_context"],
        })

df_runs = pl.DataFrame(rows)

In [22]:
df_runs = df_runs.with_columns([
    pl.col("llm").cast(pl.Boolean),
])

In [23]:
results_by_run = {}

for run_idx, temp in (
    df_runs
    .select(["run", "temp"])
    .unique()
    .iter_rows()
):

    run_df = df_runs.filter(
        (pl.col("run") == run_idx)
        & (pl.col("temp") == temp)
    )

    results_by_run[(run_idx, temp)] = process_results(run_df)

In [24]:
summary_0, cm_0, accuracy_0, metrics_df_0 = results_by_run[(0, 0.0)]

In [25]:
cm_0

gt,true,false
bool,u32,u32
false,6,110
true,56,11


In [26]:
metrics_df_0

precision,recall,f1,TP,FP,TN,FN
f64,f64,f64,i64,i64,i64,i64
0.903226,0.835821,0.868217,56,6,110,11


In [27]:
summary_1, cm_1, accuracy_1, metrics_df_1 = results_by_run[(1, 0.0)]
cm_1

KeyError: (1, 0.0)

In [ ]:
metrics_df_1

In [ ]:
summary_2, cm_2, accuracy_2, metrics_df_2 = results_by_run[(2, 0.0)]
cm_2

In [ ]:
metrics_df_2